# Análise CRM Sales Opportunities


**Dados de pipeline de vendas B2B de uma empresa fictícia que vende hardware de computador, incluindo informações sobre contas, produtos, equipes de vendas e oportunidades de vendas.**

# Importação dos dados

In [13]:
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()

engine = create_engine(
    f'postgresql+psycopg2://{os.getenv("DB_USER")}:{os.getenv("DB_PASSWORD")}'
    f'@{os.getenv("DB_HOST")}:{os.getenv("DB_PORT")}/crm_sales_opportunities'
)

# Biblioteca do CSV

In [29]:
with engine.connect() as conn:
    query = """

    select *
    from data_dictionary;
    
    """
    resultado = conn.execute(text(query))
    df = pd.DataFrame(resultado.fetchall(), columns = resultado.keys())
df

,Table,Field,Description
0,accounts,account,Company name
1,accounts,sector,Industry
2,accounts,year_established,Year Established
3,accounts,revenue,Annual revenue (in millions of USD)
4,accounts,employees,Number of employees
5,accounts,office_location,Headquarters
6,accounts,subsidiary_of,Parent company
7,products,product,Product name
8,products,series,Product series
9,products,sales_price,Suggested retail price


# 1. Faturamento mês a mês quanto foi

In [33]:
with engine.connect() as conn:
    query = """

    -- Junho de 2017 foi o melhor mês, com US$ 884.971,00 em vendas.
    -- Março de 2017 ficou em segundo lugar com US$ 757.706,00 em vendas.
    -- Dezembro que é um mês de presentes, teve menos pedidos que Julho e Março.
    
    select 
        to_char(sp.close_date::date, 'YYYY-MM') as data_fechamento,
        round(sum(sp.close_value)::numeric, 2) as faturamento
    from sales_pipeline sp
    join sales_teams st on st.sales_agent = sp.sales_agent
    join products p on p.product = sp.product
    join accounts a on a.account = sp.account
    where sp.close_date is not null
        and sp.close_date != ''
    group by to_char(sp.close_date::date, 'YYYY-MM')
    order by to_char(sp.close_date::date, 'YYYY-MM');
    
    """
   
    resultado = conn.execute(text(query))
    df = pd.DataFrame(resultado.fetchall(), columns = resultado.keys())
display(df)
    
with open('01-faturamento-geral.sql', 'w') as f:
    f.write(query)
df.to_csv('01-faturamento-geral.csv', index=False) 

,data_fechamento,faturamento
0,2017-03,757706.00
1,2017-04,449481.00
2,2017-05,673137.00
3,2017-06,884971.00
4,2017-07,443197.00
5,2017-08,696390.00
6,2017-09,746069.00
7,2017-10,466965.00
8,2017-11,622472.00
9,2017-12,754568.00


# 2. Qual empresa mais compra

In [32]:
with engine.connect() as conn:
    query = """

    -- Kan-code é a empresa que mais adquire produtos, já somados US$ 217.560,00
    -- Eles compraram o produto GTK 500 de US$ 25.791,00 e depois só compraram produtos mais baratos.

    select
        sp.account as empresa,
        round(sum(sp.close_value)::numeric, 2) as faturamento
    from sales_pipeline sp
    join sales_teams st on st.sales_agent = sp.sales_agent
    join products p on p.product = sp.product
    join accounts a on a.account = sp.account
    group by sp.account
    order by faturamento desc
    limit 10;

    """
    
    resultado = conn.execute(text(query))
    df = pd.DataFrame(resultado.fetchall(), columns = resultado.keys())
display(df)
    
with open('02-faturamento-empresa.sql', 'w') as f:
    f.write(query)
df.to_csv('02-faturamento-empresa.csv', index=False)    

,empresa,faturamento
0,Kan-code,217560.00
1,Konex,179555.00
2,Cheers,168140.00
3,Goodsilron,139457.00
4,Condax,133344.00
5,Hottechi,124574.00
6,Xx-holding,123242.00
7,Treequote,117329.00
8,Isdom,114252.00
9,Rangreen,112675.00


# 3. Qual sede mais compra os produtos

In [34]:
with engine.connect() as conn:
    query = """

    -- Estados Unidos liderou com folga em compras de hardwares, gastando US$ 5.519.293,00
    -- Em segundo a Coréia gastando US$ 124.574,00
    -- A diferença de uma pra outra é muito grande, quase 50 vezes mais que os EUA adquire.
    -- EUA investiu muito no produto GTX Plus Pro, que custa US$ 5.482 sendo ele o segundo produto mais caro.
    
    select
        a.office_location as sede_da_empresa,
        round(sum(sp.close_value)::numeric, 2) as faturamento
    from sales_pipeline sp
    join sales_teams st on st.sales_agent = sp.sales_agent
    join products p on p.product = sp.product
    join accounts a on a.account = sp.account
    group by a.office_location
    order by faturamento desc
    limit 5;
    
    """

    resultado = conn.execute(text(query))
    df = pd.DataFrame(resultado.fetchall(), columns = resultado.keys())
display(df)
    
with open('03-sede-mais-compra.sql', 'w') as f:
    f.write(query)
df.to_csv('03-sede-mais-compra.csv', index=False)

,sede_da_empresa,faturamento
0,United States,5519293.00
1,Korea,124574.00
2,Panama,112675.00
3,Belgium,85608.00
4,Norway,83547.00


# 4. Top 10 representantes de vendas por faturamento

In [35]:
with engine.connect() as conn:
    query = """

    -- Darcel Schlecht é o representante de vendas que mais trás lucro, com US$ 380.085,00 em vendas.
    -- O produto que ele mais vendeu foi o GTX Plus Pro de US$ 5.482. 
    -- Somando apenas com esse produto US$ 176.844,00 em vendas.
    -- US$ 310.473,00 em vendas para os EUA.
    -- Em segundo lugar o James Ascencio, vendendo US$ 315.852,00 e o produto que ele mais vendeu foi o GTX Plus Pro,
    -- trazendo US$ 241.420,00 em faturamento, vendendo até melhor esse produto do que o Darcel Schlecht.
    -- James Ascencio vendeu US$ 283.294,00 para os EUA, ou seja, os 2 melhores vendedores, vendem para a sede que mais compra.

    select
        sp.sales_agent as representante_de_vendas,
        round(sum(sp.close_value)::numeric, 2) as faturamento
    from sales_pipeline sp
    join sales_teams st on st.sales_agent = sp.sales_agent
    join products p on p.product = sp.product
    join accounts a on a.account = sp.account
    group by sp.sales_agent
    order by faturamento desc
    limit 10;

    """

    resultado = conn.execute(text(query))
    df = pd.DataFrame(resultado.fetchall(), columns = resultado.keys())
display(df)
    
with open('04-top10-representante-vendas-faturamento.sql', 'w') as f:
    f.write(query)
df.to_csv('04-top10-representante-vendas-faturamento.csv', index=False)    

,representante_de_vendas,faturamento
0,Darcel Schlecht,380085.00
1,James Ascencio,315852.00
2,Vicki Laflamme,289370.00
3,Anna Snelling,275056.00
4,Hayden Neloms,272111.00
5,Cassey Cress,270052.00
6,Elease Gluck,249846.00
7,Donn Cantrell,249439.00
8,Marty Freudenburg,245137.00
9,Maureen Marcano,244025.00


# 5. Top 10 gerentes de vendas por faturamento

In [37]:
with engine.connect() as conn:
    query = """

    -- Celia Rouche é a gerente que mais vende, trazendo US$ 1.259.786,00 de faturamento.
    -- A melhor representante de vendas desse time é a Vicki Laflamme, que faturou US$ 289.370,00
    -- Mesmo sendo a gerente com o melhor time de vendedores, os vendedores que mais faturam não são do grupo dela.
    -- O representante de vendas James Ascencio, que é o segundo melhor vendedor geral, trabalha na equipe do Summer Sewald,
    -- que é o segundo melhor gerente de de vendas.
    -- O terceiro melhor gerente de vendas, o Melvin Marxen, tem no seu time o Darcel Schlecht, que é o representante,
    -- de vendas que mais gera lucro total dos representantes de vendas em geral.

    select
        st.manager as gerente_de_vendas,
        round(sum(sp.close_value)::numeric, 2) as faturamento
    from sales_pipeline sp
    join sales_teams st on st.sales_agent = sp.sales_agent
    join products p on p.product = sp.product
    join accounts a on a.account = sp.account
    group by st.manager
    order by faturamento desc
    limit 10;

    """

    resultado = conn.execute(text(query))
    df = pd.DataFrame(resultado.fetchall(), columns = resultado.keys())
display(df)

with open('05-top10-gerente-vendas-faturamento.sql', 'w') as f:
    f.write(query)
df.to_csv('05-top10-gerente-vendas-faturamento.csv', index=False)

,gerente_de_vendas,faturamento
0,Celia Rouche,1259786.00
1,Summer Sewald,1188039.00
2,Melvin Marxen,1147821.00
3,Rocco Neubert,1110252.00
4,Dustin Brinkmann,1094363.00
5,Cara Losch,694695.00


# 6. Qual escritório regional tem mais vendas

In [38]:
with engine.connect() as conn:
    query = """

    -- O escritório da região Oeste é o que mais fatura, vendendo US$ 2.447.825,00
    -- A gerente de vendas que trabalha nesse escritório é a Celia Rouche, que tem a melhor equipe em faturamento.
    -- Summer Sewald sendo o gerente segundo melhor em faturamento também trabalha nesse escritório.
    -- Isso acaba explicando o por que essa região é a que mais vende.

    select
        st.regional_office as escritorio_regional,
        round(sum(sp.close_value)::numeric, 2) as faturamento
    from sales_pipeline sp
    join sales_teams st on st.sales_agent = sp.sales_agent
    join products p on p.product = sp.product
    join accounts a on a.account = sp.account
    group by st.regional_office
    order by faturamento desc
    limit 10;

    """

    resultado = conn.execute(text(query))
    df = pd.DataFrame(resultado.fetchall(), columns = resultado.keys())
display(df)
    
with open('06-escritorio-regional-vendas.sql', 'w') as f:
    f.write(query)
df.to_csv('06-escritorio-regional-vendas.csv', index=False)    

,escritorio_regional,faturamento
0,West,2447825.00
1,Central,2242184.00
2,East,1804947.00


# 7. Top 10 representantes de vendas por quantidade vendida

In [39]:
with engine.connect() as conn:
    query = """

    -- Anna Snelling é a representante que mais vendeu em quantidade totalizando 208 vendas.
    -- Em faturamento geral ela ficou em 4° lugar, com de US$ 275.056,00.
    -- Ficando em segundo lugar o Darcel Schlecht, que no top 10 faturamento ficou em 1° lugar.
    -- Uma coisa fica mais nítido, Anna Snelling vende mais vezes, porém produtos mais baratos, já
    -- o Darcel Schlecht vende menos vezes, porém produtos mais caros.

    select
        sp.sales_agent AS representante_de_vendas,
        count(sp.opportunity_id) AS quantidade_de_vendas
    from sales_pipeline sp
    join sales_teams st on st.sales_agent = sp.sales_agent
    join products p on p.product = sp.product
    join accounts a on a.account = sp.account
    where sp.deal_stage = 'Won'
    group by sp.sales_agent
    order by quantidade_de_vendas desc
    limit 10;

    """

    resultado = conn.execute(text(query))
    df = pd.DataFrame(resultado.fetchall(), columns = resultado.keys())
display(df)
    
with open('07-top10-representante-vendas-quantidade.sql', 'w') as f:
    f.write(query)
df.to_csv('07-top10-representante-vendas-quantidade.csv', index=False) 

,representante_de_vendas,quantidade_de_vendas
0,Anna Snelling,208
1,Darcel Schlecht,189
2,Vicki Laflamme,181
3,Versie Hillebrand,176
4,Kary Hendrixson,159
5,Kami Bicknell,151
6,Jonathan Berthelot,140
7,Moses Frase,129
8,Lajuana Vencill,127
9,Maureen Marcano,126


# 8. Top 10 Representantes de vendas que venderam mais rápido

In [40]:
with engine.connect() as conn:
    query = """

    -- A representante de vendas que vende mais rápido é a Rosie Papadopoulos, com uma média de 41 dias pra fechar negócio.
    -- Ela não aparece nem no top 10 quantidades vendidas, nem no top 10 faturamento.
    -- Em segundo lugar aparece a Cecily Lampkin, com uma média de 42 dias para fechar negócio.
    -- Já em terceiro lugar, com uma média de 44 dias, temos o Elease Gluck, e ele sim aparece em 7° lugar no top 10 faturamento.
    -- Isso mostra que ele, aparentemente trás bons faturamentos e também vende rápido.
    -- Marty Freudenburg que aqui ficou em 6° demorando uma média de 48 dias pra vender, no ranking de faturamento está em 9° lugar, e
    -- também merece uma atenção, pois também trás boas vendas e mais rápido.

    select
        sp.sales_agent as representante_de_vendas,
        round(avg(sp.close_date::date - sp.engage_date::date), 0) as media_de_dias
    from sales_pipeline sp
    join sales_teams st on st.sales_agent = sp.sales_agent
    join products p on p.product = sp.product
    join accounts a on a.account = sp.account
    where sp.deal_stage = 'Won'
        and sp.close_date is not null
        and sp.close_date !=''
        and sp.engage_date is not null
        and sp.engage_date !=''
    group by sp.sales_agent
    order by media_de_dias asc
    limit 10;

    """

    resultado = conn.execute(text(query))
    df = pd.DataFrame(resultado.fetchall(), columns = resultado.keys())
display(df)
    
with open('08-top10-representante-vendas-velocidade.sql', 'w') as f:
    f.write(query)
df.to_csv('08-top10-representante-vendas-velocidade.csv', index=False)

,representante_de_vendas,media_de_dias
0,Rosie Papadopoulos,41
1,Cecily Lampkin,42
2,Elease Gluck,44
3,Daniell Hammack,47
4,Boris Faz,47
5,Marty Freudenburg,48
6,Jonathan Berthelot,50
7,Zane Levy,50
8,Kami Bicknell,50
9,Kary Hendrixson,51


# 9. Produto mais vendido

In [41]:
with engine.connect() as conn:
    query = """

    -- O produto GTX Plus Pro é o que mais trouxe faturamento (US$ 2.629.651,00) pois é o segundo produto mais caro (US$ 5.482), porém
    -- o GTX Basic foi o que mais vendeu em quantidade, com 1436 produtos vendidos, e talvez seja pelo seu preço ser o penúltimo mais barato,
    -- custando US$ 550, e por isso na ordem de faturamento ficou em 4° lugar, com US$ 499.263,00 em vendas.

    select 
        p.product as produtos,
        p.sales_price as preco,
        round(sum(sp.close_value)::numeric, 2) as faturamento,
        count(sp.close_value) as quantidade_vendida
    from sales_pipeline sp
    join sales_teams st on st.sales_agent = sp.sales_agent
    join products p on p.product = sp.product
    join accounts a on a.account = sp.account
    group by p.product, p.sales_price
    order by faturamento desc
    limit 10;

    """
    
    resultado = conn.execute(text(query))
    df = pd.DataFrame(resultado.fetchall(), columns = resultado.keys())
display(df)
    
with open('09-produto-mais-vendido.sql', 'w') as f:
    f.write(query)
df.to_csv('09-produto-mais-vendido.csv', index=False) 

,produtos,preco,faturamento,quantidade_vendida
0,GTX Plus Pro,5482,2629651.00,745
1,MG Advanced,3393,2216387.00,1084
2,GTX Plus Basic,1096,705275.00,1051
3,GTX Basic,550,499263.00,1436
4,GTK 500,26768,400612.00,25
5,MG Special,55,43768.00,1223
